Wikipedia API Setup

 

We'll use two Wikipedia API endpoints.

 

Search API - Find pages related to a topic:

 

https://en.wikipedia.org/w/api.php?action=query&format=json&list=search&srsearch=YOUR_QUERY


 

Replace YOUR_QUERY with your search term. Use + for spaces (for example, lesser+capybara).

 

Get Page API - Fetch the raw content of a page:

 

https://en.wikipedia.org/w/index.php?title=PAGE_TITLE&action=raw


 

Replace PAGE_TITLE with the exact title of the Wikipedia page.

 

You can test these APIs directly in your browser to see how they work.

In [39]:
import requests

# Wikimedia User-Agent policy: https://foundation.wikimedia.org/wiki/Policy:Wikimedia_Foundation_User-Agent_Policy
# Robot policy: https://w.wiki/4wJS (keep unauthenticated API to ≤1 concurrent, <5 req/s)
WIKIPEDIA_HEADERS = {
    'User-Agent': 'WikipediaHomeworkBot/1.0 (https://w.wiki/4wJS; homework@example.com)'
}

query = 'capybara'
response = requests.get(
    f'https://en.wikipedia.org/w/api.php?action=query&format=json&list=search&srsearch={query}',
    headers=WIKIPEDIA_HEADERS
)

In [6]:
response.json()

{'batchcomplete': '',
 'continue': {'sroffset': 10, 'continue': '-||'},
 'query': {'searchinfo': {'totalhits': 897,
   'suggestion': 'capivara',
   'suggestionsnippet': 'capivara'},
  'search': [{'ns': 0,
    'title': 'Capybara',
    'pageid': 6776,
    'size': 36987,
    'wordcount': 3717,
    'snippet': 'The <span class="searchmatch">capybara</span> or greater <span class="searchmatch">capybara</span> (Hydrochoerus hydrochaeris) is the largest living rodent, native to all countries in South America except Chile. Together',
    'timestamp': '2026-03-04T22:42:55Z'},
   {'ns': 0,
    'title': 'Lesser capybara',
    'pageid': 23188846,
    'size': 5558,
    'wordcount': 581,
    'snippet': 'The lesser <span class="searchmatch">capybara</span> (Hydrochoerus isthmius) is a large semi-aquatic rodent found in South America that has vast similarities with, yet subtle differences',
    'timestamp': '2026-01-09T22:09:03Z'},
   {'ns': 0,
    'title': 'Capybara (disambiguation)',
    'pageid': 69

In [48]:
def search_wikipedia(query: str) -> dict: 
    """
    Search Wikipedia articles using the public Wikipedia Search API.

    This function queries the Wikipedia API for a search term and retrieves a summary
    of results. Designed to be used as a tool within Large Language Model (LLM) pipelines,
    agent frameworks, or conversational systems that require dynamic Wikipedia lookups.

    Args:
        query (str): The search term to send to Wikipedia. Spaces may be included naturally.

    Returns:
        dict: The parsed JSON response from the Wikipedia API, containing article hits,
              snippet previews, and suggestion data.

    Example:
        >>> results = search_wikipedia("capybara")
        >>> print(results['query']['search'][0]['title'])

    API Reference:
        https://en.wikipedia.org/w/api.php?action=query&format=json&list=search&srsearch=YOUR_QUERY

    Requirements:
        The requests package must be installed. The function automatically sets a proper 
        User-Agent as required by the API.

    LLM Usage:
        This tool can be registered as a function calling tool in LLM products such as 
        OpenAI function calls, LangChain Tools, or similar agent frameworks. It is safe 
        for external API access and returns fully JSON-compatible results.
    """
    query = query.replace(' ', '+')
    response = requests.get(
        f'https://en.wikipedia.org/w/api.php?action=query&format=json&list=search&srsearch={query}',
        headers=WIKIPEDIA_HEADERS
    )
    return response.json()

json_output = search_wikipedia('capybara')



In [ ]:
## Qyestion 1 - How many results did you get?
len(json_output['query']['search'])

10

In [ ]:
## Qyestion 2 - Cases that contain the word "capybara" in the title case insensitive
cases_with_capybara = [x['title'].lower().find('capybara')>-1 for x in json_output['query']['search']]
sum(cases_with_capybara)



5

In [47]:
def get_page(title: str) -> str:
    """Fetch raw wikitext of a Wikipedia page. Uses WIKIPEDIA_HEADERS per robot policy."""
    title = title.replace(' ', '+')
    url = f'https://en.wikipedia.org/w/index.php?title={title}&action=raw'
    response = requests.get(url, headers=WIKIPEDIA_HEADERS)
    return response.text

page = get_page('Capybara')
print(page)



{{Short description|Largest species of rodents}}
{{Other uses}}
{{Good article}}
{{pp|small=yes}}
{{Use dmy dates|date=July 2022}}
{{Speciesbox
| status = LC
| status_system = IUCN3.1
| status_ref = <ref name="iucn status 19 November 2021">{{cite iucn |author=Reid, F. |date=2016 |title=''Hydrochoerus hydrochaeris'' |volume=2016 |article-number=e.T10300A22190005 |doi=10.2305/IUCN.UK.2016-2.RLTS.T10300A22190005.en |access-date=19 November 2021}}</ref>
| image = 046 Capybara by the river in Encontro das Águas State Park Photo by Giles Laurent.jpg
| image_caption = In [[Encontro das Águas State Park]], Brazil
| genus = Hydrochoerus
| species = hydrochaeris
| authority = ([[Carl Linnaeus|Linnaeus]], [[12th edition of Systema Naturae|1766]])
| range_map = Capybara range.svg
| range_map_caption = Native range
| synonyms = ''Sus hydrochaeris'' {{small|Linnaeus,&nbsp;1766}}
}}

The '''capybara'''{{efn | Also referred as '''capivara''' (in Brazil), '''capiguara''' (in Bolivia), '''chigüire''', '

In [41]:
## Qyestion 3 - How many characters does the page have?
len(page)

36863

In [61]:
from pydantic_ai import Agent

agent = Agent(
    model="gpt-4o-mini",
    name="wikipedia_agent",
    instructions="You are a helpful assistant that can search the web and retrieve information from Wikipedia.",
    tools=[search_wikipedia, get_page])       


In [62]:
query = "What is this page about? https://en.wikipedia.org/wiki/Capybara"
response = await agent.run(query) 

In [64]:
print(response.output)

The Wikipedia page on the **capybara** (scientific name: *Hydrochoerus hydrochaeris*) provides information about this animal, which is the largest living rodent in the world, native to South America (excluding Chile). The capybara is known for its social nature and tends to live in groups of 10 to 20 individuals, but can sometimes form groups of up to 100.

Key points from the page include:

- **Description**: Capybaras have a heavy, barrel-shaped body with reddish-brown fur on the top and yellowish-brown underneath. They can grow between 106 to 134 cm in length and weigh between 35 to 66 kg. They are semiaquatic creatures that thrive near bodies of water.

- **Habitat**: They can be found in savannas and dense forests, often in proximity to lakes, rivers, and swamps.

- **Diet**: Capybaras are herbivores and mainly feed on grasses, aquatic plants, and certain fruits. They have a unique digestive system that allows them to extract maximum nutrients from their fibrous diet by re-eating 

## Streaming the agent response

Use an **event stream handler** to stream model text as it’s generated and optionally log tool calls. Pass it when calling `run()` with `event_stream_handler=...`, or set it on the agent with `event_stream_handler=...`.

In [58]:
from pydantic_ai.messages import FunctionToolCallEvent


async def _handle_stream_event(ctx, event):
    """Handle a single event (may be nested async iterable)."""
    if hasattr(event, "__aiter__"):
        async for sub in event:
            await _handle_stream_event(ctx, sub)
    elif isinstance(event, FunctionToolCallEvent):
        print(f"\n[Tool: {event.part.tool_name}({event.part.args})]\n", flush=True)


async def wikipedia_stream_handler(ctx, stream):
    """Stream model text token-by-token and optionally print tool calls."""
    # Model request node: stream is AgentStream with stream_text()
    if hasattr(stream, "stream_text"):
        async for chunk in stream.stream_text(delta=True, debounce_by=0.02):
            print(chunk, end="", flush=True)
        return
    # Tool-call node: stream is async iterable of events (e.g. FunctionToolCallEvent)
    async for event in stream:
        await _handle_stream_event(ctx, event)


# Run with streaming (text appears as it’s generated)
query ="What are the main threats to capybara populations?" 
result = await agent.run(query, event_stream_handler=wikipedia_stream_handler)
print("\n--- Done ---")


[Tool: search_wikipedia({"query":"Capybara threats to populations"})]


[Tool: search_wikipedia({"query":"Capybara"})]


[Tool: get_page({"title":"Capybara"})]

The main threats to capybara populations include:

1. **Hunting**: Capybaras are hunted for their meat and pelts in some areas of South America. Their hides are valuable, and they are also sought after for food, especially during specific cultural events such as Lent in Venezuela.

2. **Habitat Loss**: The expansion of agriculture and urbanization reduces the natural habitats of capybaras. As human communities expand into wetlands, the availability of water and grazing areas for capybaras decreases.

3. **Competition with Livestock**: In some regions, capybaras are perceived as competition for resources with livestock, leading to them being killed or driven away by farmers.

4. **Predation**: Although capybaras have natural predators such as jaguars, caimans, and anacondas, increased human activities can impact these predator-

In [59]:
print(response.output)

The Wikipedia page on the **capybara** provides detailed information about this species, which is known as the largest living rodent and is scientifically named *Hydrochoerus hydrochaeris*. Here are some key points from the page:

1. **Habitat and Distribution**: 
   - Capybaras are native to South America and live in habitats that include savannas and dense forests, often near bodies of water. They can be found in all South American countries except Chile.

2. **Social Behavior**:
   - They are highly social animals, typically found in groups ranging from 10 to 20, but can gather in larger groups of up to 100 during dry seasons.

3. **Physical Description**:
   - Capybaras have a heavy, barrel-shaped body, short head, and reddish-brown fur, with body lengths of about 106 to 134 cm and weights ranging from 35 to 66 kg.

4. **Diet**:
   - As herbivores, capybaras primarily feed on grasses, but they also eat aquatic plants, fruit, and tree bark. They practice coprophagy, consuming their 

In [60]:
response.all_messages()

[ModelRequest(parts=[UserPromptPart(content='What is this page about? https://en.wikipedia.org/wiki/Capybara', timestamp=datetime.datetime(2026, 3, 5, 23, 55, 11, 360750, tzinfo=datetime.timezone.utc))], timestamp=datetime.datetime(2026, 3, 5, 23, 55, 11, 360873, tzinfo=datetime.timezone.utc), instructions='You are a helpful assistant that can search the web and retrieve information from Wikipedia.', run_id='eb17d5d9-3a71-47e3-97ae-235cbcb93b69'),
 ModelResponse(parts=[ToolCallPart(tool_name='get_page', args='{"title":"Capybara"}', tool_call_id='call_KE6beKezL4jkQJhwbRCtZ7en')], usage=RequestUsage(input_tokens=220, output_tokens=16, details={'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}), model_name='gpt-4o-mini-2024-07-18', timestamp=datetime.datetime(2026, 3, 5, 23, 55, 12, 963660, tzinfo=datetime.timezone.utc), provider_name='openai', provider_url='https://api.openai.com/v1/', provider_details={'finish_reason': 'tool_call

In [ ]:
## Qyestion 4 - What is the page about?